# Метод опорных векторов (SVM) — практическое задание

В этом ноутбуке рассматривается метод **опорных векторов (Support Vector Machine, SVM)** на примере набора данных **Breast Cancer Wisconsin (Diagnostic)** с Kaggle.
Ноутбук рассчитан на выполнение в **Google Colab**.


## Теоретическое введение

### 1. Постановка задачи бинарной классификации

Пусть имеется обучающая выборка
$$
\{(x_i, y_i)\}_{i=1}^n, \qquad x_i \in \mathbb{R}^d,\quad y_i \in \{-1,+1\}.
$$

Здесь:
- $x_i$ — вектор признаков объекта;
- $y_i$ — метка класса.

Требуется построить правило классификации, которое по новому вектору признаков $x$ определяет его класс.

### 2. Разделяющая гиперплоскость

В линейном случае SVM ищет гиперплоскость вида
$$
w^T x + b = 0,
$$
где:
- $w \in \mathbb{R}^d$ — вектор параметров,
- $b \in \mathbb{R}$ — свободный член.

Классификация выполняется по знаку выражения:
$$
\hat y = \operatorname{sign}(w^T x + b).
$$

### 3. Идея максимального зазора

Если классы линейно разделимы, существует много гиперплоскостей, разделяющих данные.  
SVM выбирает такую гиперплоскость, у которой **зазор** (margin) между классами максимален.

При нормировке условий классификации:
$$
y_i(w^T x_i + b) \ge 1,\qquad i=1,\dots,n,
$$
ширина зазора равна
$$
\frac{2}{\|w\|}.
$$

Следовательно, задача максимизации зазора эквивалентна задаче минимизации нормы вектора $w$:
$$
\min_{w,b}\frac{1}{2}\|w\|^2
\quad \text{при условиях}\quad
y_i(w^T x_i + b)\ge 1.
$$

### 4. Мягкий зазор

На практике классы часто не разделяются идеально. Тогда вводятся переменные ошибок $\xi_i \ge 0$:
$$
y_i(w^T x_i + b) \ge 1 - \xi_i.
$$

Получаем задачу:
$$
\min_{w,b,\xi}\frac{1}{2}\|w\|^2 + C\sum_{i=1}^n \xi_i,
$$
где параметр $C>0$ управляет компромиссом между:
- большим зазором;
- числом и величиной ошибок классификации.

Если $C$ велико, модель сильнее штрафует ошибки.  
Если $C$ меньше, модель допускает больше нарушений, но может лучше обобщать.

### 5. Ядровой метод

Если данные не разделяются линейно в исходном пространстве признаков, можно применить отображение
$$
x \mapsto \phi(x)
$$
в пространство большей размерности и строить разделяющую гиперплоскость уже там.

На практике явное отображение часто не требуется. Используется **ядро**
$$
K(x,z)=\langle \phi(x), \phi(z)\rangle.
$$

Популярные варианты:
- линейное ядро;
- полиномиальное ядро;
- радиально-базисное ядро (RBF).

### 6. Почему для SVM важно масштабирование

SVM чувствителен к масштабу признаков, поскольку расстояния и скалярные произведения напрямую участвуют в построении разделяющей поверхности.  
Если признаки имеют сильно разные масштабы, модель может работать некорректно. Поэтому перед обучением часто применяют стандартизацию:
$$
x_j^{(\mathrm{scaled})}=\frac{x_j-\mu_j}{\sigma_j}.
$$

### 7. Что оценивают после обучения

После обучения модели обычно анализируют:
- точность классификации (`accuracy`);
- матрицу ошибок (`confusion matrix`);
- `precision`, `recall`, `f1-score`;
- влияние параметров `C` и типа ядра на качество модели.


## Набор данных

Для работы используется набор данных **Breast Cancer Wisconsin (Diagnostic)**:
[https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data](https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data).

Целевая переменная:
- `diagnosis` — диагноз (`M` = malignant, злокачественная опухоль; `B` = benign, доброкачественная опухоль).

Примеры признаков:
- `radius_mean`;
- `texture_mean`;
- `perimeter_mean`;
- `area_mean`;
- `smoothness_mean`;
- и другие числовые характеристики клеточных ядер.

Этот набор данных хорошо подходит для SVM, потому что:
- задача является бинарной классификацией;
- признаки в основном числовые;
- данные хорошо подходят для исследования линейной и нелинейной границы;
- можно на практике увидеть важность масштабирования признаков.

Самостоятельно выберите способ загрузить данные и сделать их доступными в блокноте с заданием.

Один из вариантов — скачивание датасета напрямую из Kaggle через API в Google Colab.


## Как загрузить данные с Kaggle прямо в Google Colab

Выполните предварительную настройку Kaggle и Colab (делается один раз).
1. Откройте https://www.kaggle.com, зайдите в свой профиль (создайте его, если не создали ранее), затем в настройках профиля найдите раздел **API Tokens (Recommended)**.
2. Создайте токен и сразу же скопируйте его в текстовый документ на своём локальном компьютере.
3. Там же посмотрите, под каким именем пользователя (`username`) вы работаете в Kaggle. Скопируйте `username` в тот же документ.
4. В Colab откройте левую панель, вкладку **Secrets**.
5. Создайте два секрета:
   - `KAGGLE_USERNAME` : вставьте `username`;
   - `KAGGLE_KEY`      : вставьте `token`.

Не забудьте установить переключатель **«Доступ из блокнотов»**.

Далее следуйте инструкциям в коде ниже:


In [ ]:
# ШАГ 1. Установка Kaggle API
!pip -q install kaggle

# ШАГ 2. Чтение токена из Colab Secrets
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

# ШАГ 3. Проверка, что всё работает
!kaggle datasets list -s "breast cancer wisconsin" | head

# ШАГ 4. Скачивание датасета
TARGET_DIR = "/content/sci-prog-and-ml"
!mkdir -p {TARGET_DIR}
os.chdir(TARGET_DIR)

# Настройка адреса скачивания: берём из URL всё, что идёт
# после /datasets/ : https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data
DATASET = "uciml/breast-cancer-wisconsin-data"
!kaggle datasets download -d {DATASET} -p {TARGET_DIR} --unzip

# ШАГ 5. Проверка содержимого папки
print("Файлы в рабочей папке:")
for file in os.listdir(TARGET_DIR):
    print(" -", file)


## Альтернативный вариант — скачивание файла вручную

- создайте у себя на Google-диске папку `sci-prog-and-ml`;
- скачайте файл данных вручную и поместите его в эту папку на Google-диске;
- подмонтируйте Google-диск к среде Colab, выполнив код ниже (ответьте утвердительно на запросы о предоставлении доступа).

После выполнения этих инструкций папка `sci-prog-and-ml` будет сделана рабочей и из неё уже можно будет открывать файл при помощи `df = pd.read_csv(file_name)`.


In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/sci-prog-and-ml')

print("Содержимое текущей папки:")
for file in os.listdir('.'):
    print(" -", file)


## Задание

### Часть 1. Загрузка и первичный анализ данных
1. Загрузите файл с данными в `pandas.DataFrame`.
2. Выведите первые строки таблицы.
3. Определите:
   - число объектов;
   - список признаков;
   - типы столбцов;
   - наличие пропущенных значений.

### Часть 2. Подготовка данных
1. Удалите служебные столбцы, если они есть (например, `id`, `Unnamed: 32`).
2. Выделите целевую переменную `diagnosis`.
3. Преобразуйте её в числовой вид:
   - `M -> 1`,
   - `B -> 0`.
4. Сформируйте матрицу признаков `X` и вектор ответов `y`.
5. Разделите выборку на обучающую и тестовую части.
6. Выполните стандартизацию признаков с помощью `StandardScaler`.

### Часть 3. Построение модели SVM
1. Создайте модель `SVC` с линейным ядром.
2. Обучите модель на обучающей выборке.
3. Получите предсказания на тестовой выборке.
4. Вычислите:
   - `accuracy`,
   - матрицу ошибок,
   - `classification_report`.

### Часть 4. Исследование параметров
1. Постройте ещё одну модель SVM с ядром `rbf`.
2. Сравните качество линейного и RBF-варианта.
3. Измените параметр `C` и исследуйте, как он влияет на результат.
4. Сделайте вывод о том, какая модель лучше работает на данном наборе данных.


## Подсказки

### Основные библиотеки
```python
import os
import glob
import pandas as pd
import numpy as np
```

### Загрузка данных

В датасете `uciml/breast-cancer-wisconsin-data` после распаковки обычно появляется CSV-файл с данными.  
Чтобы код не зависел от точного имени файла, удобно автоматически найти первый CSV-файл в рабочей папке:

```python
csv_files = glob.glob("*.csv")
print(csv_files)

df = pd.read_csv(csv_files[0])
```

Если вы хотите указать имя файла явно, сначала посмотрите содержимое рабочей папки:
```python
import os
print(os.listdir())
```

### Удаление лишних столбцов
```python
for col in ["id", "Unnamed: 32"]:
    if col in df.columns:
        df = df.drop(columns=col)
```

### Кодирование целевой переменной
```python
y = df["diagnosis"].map({"M": 1, "B": 0})
X = df.drop(columns=["diagnosis"])
```

### Разделение на обучающую и тестовую выборки
```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
```

### Масштабирование признаков
```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
```

### Модель SVM
```python
from sklearn.svm import SVC

model = SVC(kernel="linear", C=1.0, random_state=42)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
```

### Метрики качества
```python
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
```


## Решение


In [ ]:
# Поместите Ваш код сюда


## Вопросы для обсуждения

1. Почему для SVM важно масштабирование признаков?
2. Что означает максимизация зазора между классами?
3. Как влияет параметр `C` на разделяющую поверхность и обобщающую способность модели?
4. В чём различие между линейным SVM и SVM с RBF-ядром?
5. Почему даже при высокой точности полезно анализировать матрицу ошибок и `precision/recall`?


## Дополнительные задания

### Задание 1
Постройте несколько моделей SVM с разными значениями `C`, например:
- `0.1`,
- `1`,
- `10`,
- `100`.

Сравните качество моделей и сделайте вывод.

### Задание 2
Для модели с RBF-ядром дополнительно исследуйте параметр `gamma`.

### Задание 3
Попробуйте обучить SVM **без масштабирования признаков** и сравните результат с масштабированным вариантом.

### Задание 4
Выберите два признака, постройте двумерную версию задачи и визуализируйте разделяющую границу.
